# 10. Productivizar el Modelo

In [81]:
# ── Comprobación del split ──
assert TRAIN_AÑOS == [2018, 2019, 2020, 2021, 2022], f"TRAIN_AÑOS inesperado: {TRAIN_AÑOS}"
assert TEST_AÑOS  == [2023, 2024],                   f"TEST_AÑOS inesperado: {TEST_AÑOS}"

# ── Modelo final: Random Forest──

MODELO_FINAL = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
).fit(X_tr, y_tr)

NOMBRE_MODELO  = "Random Forest"
UMBRAL_FINAL   = 0.50
FEATURES_FINAL = list(FEATURES_AVANZADO_DISP)

# ── Métricas reales del objeto que se va a empaquetar ──
_proba = MODELO_FINAL.predict_proba(X_te)[:, 1]
_pred  = (_proba >= UMBRAL_FINAL).astype(int)
_cm    = confusion_matrix(y_te, _pred)

METRICAS_FINAL = {
    "f1":          round(float(f1_score(y_te, _pred)), 3),
    "recall":      round(float(recall_score(y_te, _pred)), 3),
    "especificidad": round(float(_cm[0, 0] / _cm[0].sum()), 3),
    "f1_macro":    round(float(f1_score(y_te, _pred, average="macro")), 3),
    "auc_roc":     round(float(roc_auc_score(y_te, _proba)), 3),
    "n_train":     int(len(X_tr)),
    "n_test":      int(len(X_te)),
    "años_train":  list(TRAIN_AÑOS),
    "años_test":   list(TEST_AÑOS),
    "n_features":  len(FEATURES_FINAL),
    "umbral":      UMBRAL_FINAL,
    "matriz_confusion": {
        "no_prioritario_correctos": int(_cm[0, 0]),
        "falsos_positivos":         int(_cm[0, 1]),
        "falsos_negativos":         int(_cm[1, 0]),
        "prioritarios_detectados":  int(_cm[1, 1]),
    },
}

print("=" * 64)
print("  MODELO QUE SE VA A EMPAQUETAR")
print("=" * 64)
print(f"  Algoritmo : {NOMBRE_MODELO} ({type(MODELO_FINAL).__name__})")
print(f"  Features  : {len(FEATURES_FINAL)}")
print(f"  Umbral    : {UMBRAL_FINAL}")
print(f"  Train     : {TRAIN_AÑOS[0]}–{TRAIN_AÑOS[-1]}  ({len(X_tr):,} filas)")
print(f"  Test      : {TEST_AÑOS[0]}–{TEST_AÑOS[-1]}  ({len(X_te):,} filas)")
print(f"\n  Recall (prioritario) : {METRICAS_FINAL['recall']}")
print(f"  Especificidad        : {METRICAS_FINAL['especificidad']}")
print(f"  F1-macro             : {METRICAS_FINAL['f1_macro']}")
print(f"  AUC-ROC              : {METRICAS_FINAL['auc_roc']}")
print(f"\n  Prioritarios detectados: {_cm[1,1]}/{_cm[1].sum()}   "
      f"No prioritarios correctos: {_cm[0,0]}/{_cm[0].sum()}")


  MODELO QUE SE VA A EMPAQUETAR
  Algoritmo : Random Forest (RandomForestClassifier)
  Features  : 36
  Umbral    : 0.5
  Train     : 2018–2022  (2,042 filas)
  Test      : 2023–2024  (1,566 filas)

  Recall (prioritario) : 0.758
  Especificidad        : 0.692
  F1-macro             : 0.704
  AUC-ROC              : 0.812

  Prioritarios detectados: 410/541   No prioritarios correctos: 709/1025


**Artefactos, tabla de referencia y metadata**

In [82]:
RUTA_DOCKER.mkdir(parents=True, exist_ok=True)
RUTA_MODELOS.mkdir(parents=True, exist_ok=True)

# ── 1. Los cuatro artefactos que app.py carga al arrancar ──
joblib.dump(MODELO_FINAL,        RUTA_DOCKER / "modelo_binario_final.pkl")
joblib.dump(FEATURES_FINAL,      RUTA_DOCKER / "features_binario.pkl")
joblib.dump(le_bin,              RUTA_DOCKER / "encoder_binario.pkl")
joblib.dump(float(UMBRAL_FINAL), RUTA_DOCKER / "umbral_optimo.pkl")

# ── 2. Tabla de referencia: una fila por municipio-año ──
cols_tabla = ["cod_mun", "año"] + [f for f in FEATURES_FINAL if f != "año"]
if "NOMBRE" in df_bin.columns:
    cols_tabla.insert(2, "NOMBRE")

tabla_ref = (
    df_bin[cols_tabla]
    .dropna(subset=FEATURES_FINAL)
    .drop_duplicates(subset=["cod_mun", "año"], keep="last")
    .sort_values(["cod_mun", "año"])
    .reset_index(drop=True)
)
tabla_ref["cod_mun"] = tabla_ref["cod_mun"].astype(str).str.zfill(5)
tabla_ref["año"]     = tabla_ref["año"].astype(int)
tabla_ref.to_csv(RUTA_DOCKER / "municipios_features.csv", index=False, encoding="utf-8")

# ── 3. Metadata: versiones, hiperparámetros y métricas ──
metadata = {
    "modelo":          NOMBRE_MODELO,
    "clase":           type(MODELO_FINAL).__name__,
    "hiperparametros": {
        k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v))
        for k, v in MODELO_FINAL.get_params().items()
    },
    "features":        FEATURES_FINAL,
    "clases":          list(le_bin.classes_),
    "umbral_decision": UMBRAL_FINAL,
    "metricas_test":   METRICAS_FINAL,
    "entrenado_en":    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "versiones": {
        "python":       f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
        "scikit-learn": sklearn.__version__,
        "numpy":        np.__version__,
        "pandas":       pd.__version__,
        "joblib":       joblib.__version__,
    },
    "cobertura_tabla": {
        "municipios": int(tabla_ref["cod_mun"].nunique()),
        "años":       sorted(int(a) for a in tabla_ref["año"].unique()),
        "filas":      int(len(tabla_ref)),
    },
}
with open(RUTA_DOCKER / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# ── 4. Respaldo en RUTA_MODELOS ──
for archivo in ["modelo_binario_final.pkl", "features_binario.pkl",
                "encoder_binario.pkl", "umbral_optimo.pkl", "metadata.json"]:
    shutil.copy(RUTA_DOCKER / archivo, RUTA_MODELOS / archivo)

print("=== ARTEFACTOS GUARDADOS ===\n")
for archivo in sorted(RUTA_DOCKER.iterdir()):
    if archivo.is_file():
        print(f"  ✓ {archivo.name:<32} {archivo.stat().st_size/1024:>9.1f} KB")

print(f"\n  Tabla de referencia: {len(tabla_ref):,} filas · "
      f"{tabla_ref['cod_mun'].nunique():,} municipios · "
      f"años {metadata['cobertura_tabla']['años']}")

_perdidas = len(df_bin) - len(tabla_ref)
if _perdidas > 0:
    print(f"  ⚠ {_perdidas:,} filas de df_bin quedaron fuera por NaN en alguna feature")




=== ARTEFACTOS GUARDADOS ===

  ✓ .dockerignore                          0.1 KB
  ✓ Dockerfile                             1.0 KB
  ✓ README.md                              4.6 KB
  ✓ app.py                                10.1 KB
  ✓ encoder_binario.pkl                    0.4 KB
  ✓ features_binario.pkl                   0.7 KB
  ✓ metadata.json                          2.3 KB
  ✓ modelo_binario_final.pkl            2016.5 KB
  ✓ municipios_features.csv             1064.2 KB
  ✓ requirements.txt                       0.1 KB
  ✓ test_api.py                            3.1 KB
  ✓ umbral_optimo.pkl                      0.0 KB

  Tabla de referencia: 3,608 filas · 914 municipios · años [2020, 2021, 2022, 2023, 2024]
  ⚠ 2,722 filas de df_bin quedaron fuera por NaN en alguna feature


**app.py**

In [83]:
app_code = '''"""
API — Predicción de riesgo de desnutrición aguda infantil en municipios de Colombia.
TFM · Máster en Data Science · Jefferson Montoya Hoyos
"""
from pathlib import Path
from typing import Dict, List, Optional

import joblib
import json
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field

BASE = Path(__file__).parent

# ── Carga de artefactos al arrancar ──
modelo   = joblib.load(BASE / "modelo_binario_final.pkl")
features = joblib.load(BASE / "features_binario.pkl")
encoder  = joblib.load(BASE / "encoder_binario.pkl")
umbral   = float(joblib.load(BASE / "umbral_optimo.pkl"))
metadata = json.loads((BASE / "metadata.json").read_text(encoding="utf-8"))

tabla = pd.read_csv(BASE / "municipios_features.csv", dtype={"cod_mun": str})
tabla["cod_mun"] = tabla["cod_mun"].str.zfill(5)
tabla["año"] = tabla["año"].astype(int)

# Si falta una feature en la tabla, mejor fallar al arrancar que en la primera
# petición: así `docker run` muestra el error de inmediato.
_faltantes = [f for f in features if f not in tabla.columns]
if _faltantes:
    raise RuntimeError(f"municipios_features.csv no contiene: {_faltantes}")

app = FastAPI(
    title="API — Predicción de Riesgo de Desnutrición Infantil",
    description=(
        "Clasifica municipios de Colombia como **prioritario** o "
        "**no_prioritario** según el riesgo de desnutrición aguda en menores "
        "de 5 años. TFM · Máster en Data Science."
    ),
    version="1.1.0",
)


# ══════════════════════════════════════════════════════════════════
# ESQUEMAS
# ══════════════════════════════════════════════════════════════════
class ConsultaMunicipio(BaseModel):
    cod_mun: str = Field("44847", description="Código DIVIPOLA de 5 dígitos")
    anio: int = Field(2024, alias="año", description="Año a evaluar")

    model_config = {
        "populate_by_name": True,
        "json_schema_extra": {"example": {"cod_mun": "44847", "año": 2024}},
    }


class PrediccionOutput(BaseModel):
    cod_mun: str
    nombre: Optional[str] = None
    anio: int
    clasificacion: str
    probabilidad: float
    umbral_usado: float
    intervencion: bool
    mensaje_negocio: str


class EscenarioManual(BaseModel):
    """Simulación what-if: se pasan las features directamente."""
    valores: Dict[str, float] = Field(
        ..., description="Diccionario feature -> valor. Debe traer todas las del modelo."
    )
    etiqueta: str = "escenario_manual"


# ══════════════════════════════════════════════════════════════════
# LÓGICA
# ══════════════════════════════════════════════════════════════════
def _buscar_fila(cod_mun: str, anio: int) -> pd.Series:
    cod = str(cod_mun).zfill(5)
    fila = tabla[(tabla["cod_mun"] == cod) & (tabla["año"] == anio)]
    if fila.empty:
        disponibles = sorted(tabla.loc[tabla["cod_mun"] == cod, "año"].unique().tolist())
        if disponibles:
            raise HTTPException(
                404, f"No hay datos para {cod} en {anio}. Años disponibles: {disponibles}"
            )
        raise HTTPException(404, f"Municipio {cod} no encontrado en la base.")
    return fila.iloc[0]


def _predecir_df(X: pd.DataFrame) -> np.ndarray:
    return modelo.predict_proba(X[features].astype(float))[:, 1]


def _mensaje(cod: str, prob: float, prioritario: bool) -> str:
    if prioritario:
        return (
            f"El municipio {cod} presenta ALTA probabilidad de riesgo de "
            f"desnutrición aguda infantil ({prob:.1%}). Se recomienda intervención "
            "prioritaria: brigadas de salud, refuerzo de programas nutricionales "
            "y seguimiento por parte de la autoridad sanitaria."
        )
    return (
        f"El municipio {cod} presenta probabilidad baja-media de riesgo "
        f"({prob:.1%}). Monitoreo de rutina mediante el sistema SIVIGILA."
    )


# ══════════════════════════════════════════════════════════════════
# ENDPOINTS
# ══════════════════════════════════════════════════════════════════
@app.get("/", tags=["Estado"])
def root():
    return {
        "api": "Predicción de Riesgo de Desnutrición Infantil",
        "version": app.version,
        "estado": "activa",
        "docs": "/docs",
    }


@app.get("/salud", tags=["Estado"])
def salud():
    return {
        "estado": "activo",
        "modelo": metadata["modelo"],
        "clase": type(modelo).__name__,
        "n_features": len(features),
        "umbral_decision": round(umbral, 3),
        "clases": list(encoder.classes_),
        "metricas_test": metadata["metricas_test"],
        "versiones": metadata["versiones"],
        "municipios_en_base": int(tabla["cod_mun"].nunique()),
        "años_disponibles": sorted(int(a) for a in tabla["año"].unique()),
    }


@app.get("/features", tags=["Consulta"])
def lista_features():
    """Features del modelo y su orden exacto."""
    return {"n_features": len(features), "features": features}


@app.get("/municipios", tags=["Consulta"])
def municipios(
    anio: Optional[int] = Query(None, alias="año"),
    limite: int = Query(50, ge=1, le=2000),
):
    """Municipios disponibles, opcionalmente filtrados por año."""
    df = tabla if anio is None else tabla[tabla["año"] == anio]
    cols = ["cod_mun", "año"] + (["NOMBRE"] if "NOMBRE" in df.columns else [])
    return {
        "total": int(len(df)),
        "municipios": df[cols].head(limite).to_dict(orient="records"),
    }


@app.post("/predecir", response_model=PrediccionOutput, tags=["Predicción"])
def predecir(consulta: ConsultaMunicipio):
    """Clasifica un municipio a partir de su código DIVIPOLA y el año."""
    fila = _buscar_fila(consulta.cod_mun, consulta.anio)
    X = fila[features].to_frame().T
    prob = float(_predecir_df(X)[0])
    prioritario = prob >= umbral

    return PrediccionOutput(
        cod_mun=str(fila["cod_mun"]),
        nombre=str(fila["NOMBRE"]) if "NOMBRE" in fila.index else None,
        anio=int(consulta.anio),
        clasificacion="prioritario" if prioritario else "no_prioritario",
        probabilidad=round(prob, 4),
        umbral_usado=round(umbral, 3),
        intervencion=prioritario,
        mensaje_negocio=_mensaje(str(fila["cod_mun"]), prob, prioritario),
    )


@app.post("/predecir/lote", tags=["Predicción"])
def predecir_lote(consultas: List[ConsultaMunicipio]):
    """Clasifica varios municipios en una sola llamada (máx. 500)."""
    if len(consultas) > 500:
        raise HTTPException(400, "Máximo 500 municipios por llamada")

    filas, no_encontrados = [], []
    for c in consultas:
        cod = str(c.cod_mun).zfill(5)
        f = tabla[(tabla["cod_mun"] == cod) & (tabla["año"] == c.anio)]
        if f.empty:
            no_encontrados.append({"cod_mun": cod, "año": c.anio})
        else:
            filas.append(f.iloc[0])

    if not filas:
        raise HTTPException(404, "Ningún municipio de la lista fue encontrado.")

    X = pd.DataFrame(filas)
    probs = _predecir_df(X)

    resultados = [
        {
            "cod_mun": str(r["cod_mun"]),
            "nombre": str(r["NOMBRE"]) if "NOMBRE" in r.index else None,
            "año": int(r["año"]),
            "probabilidad": round(float(p), 4),
            "clasificacion": "prioritario" if p >= umbral else "no_prioritario",
            "intervencion": bool(p >= umbral),
        }
        for r, p in zip(filas, probs)
    ]
    resultados.sort(key=lambda x: x["probabilidad"], reverse=True)

    return {
        "total": len(resultados),
        "prioritarios": sum(1 for r in resultados if r["intervencion"]),
        "no_prioritarios": sum(1 for r in resultados if not r["intervencion"]),
        "no_encontrados": no_encontrados,
        "ranking": resultados,
    }


@app.get("/ranking", tags=["Predicción"])
def ranking(anio: int = Query(..., alias="año"), top: int = Query(20, ge=1, le=500)):
    """Ranking nacional de municipios por probabilidad de riesgo en un año."""
    df = tabla[tabla["año"] == anio]
    if df.empty:
        disponibles = sorted(int(a) for a in tabla["año"].unique())
        raise HTTPException(404, f"No hay datos para {anio}. Disponibles: {disponibles}")

    probs = _predecir_df(df)
    cols = ["cod_mun"] + (["NOMBRE"] if "NOMBRE" in df.columns else [])
    out = df[cols].copy()
    out["probabilidad"] = np.round(probs, 4)
    out["clasificacion"] = np.where(probs >= umbral, "prioritario", "no_prioritario")
    out = out.sort_values("probabilidad", ascending=False).head(top)

    return {
        "año": anio,
        "total_municipios": int(len(df)),
        "prioritarios_en_el_año": int((probs >= umbral).sum()),
        "top": out.to_dict(orient="records"),
    }


@app.post("/predecir/manual", tags=["Predicción"])
def predecir_manual(escenario: EscenarioManual):
    """Simulación what-if con las features pasadas a mano."""
    faltan = [f for f in features if f not in escenario.valores]
    if faltan:
        raise HTTPException(
            422,
            f"Faltan {len(faltan)} features: {faltan[:10]}"
            f"{' ...' if len(faltan) > 10 else ''}. Consulta GET /features.",
        )
    X = pd.DataFrame([{f: float(escenario.valores[f]) for f in features}])
    prob = float(_predecir_df(X)[0])
    prioritario = prob >= umbral
    return {
        "etiqueta": escenario.etiqueta,
        "probabilidad": round(prob, 4),
        "clasificacion": "prioritario" if prioritario else "no_prioritario",
        "umbral_usado": round(umbral, 3),
        "intervencion": prioritario,
    }
'''

with open(RUTA_DOCKER / "app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
print(f"✓ app.py  ({len(app_code):,} caracteres)  →  {RUTA_DOCKER}")




✓ app.py  (9,515 caracteres)  →  /content/drive/MyDrive/Máster Data Science/TFM/docker


**requirements.txt con las versiones reales del entorno**

In [84]:
reqs = f"""fastapi==0.115.6
uvicorn[standard]==0.34.0
pydantic==2.10.4
scikit-learn=={sklearn.__version__}
pandas=={pd.__version__}
numpy=={np.__version__}
joblib=={joblib.__version__}
"""
with open(RUTA_DOCKER / "requirements.txt", "w", encoding="utf-8") as f:
    f.write(reqs)

print("✓ requirements.txt:\n")
print(reqs)




✓ requirements.txt:

fastapi==0.115.6
uvicorn[standard]==0.34.0
pydantic==2.10.4
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.1.3
joblib==1.6.0



**Dockerfile y .dockerignore**

In [85]:
PY_TAG = f"{sys.version_info.major}.{sys.version_info.minor}"

dockerfile = f"""FROM python:{PY_TAG}-slim

# Sin .pyc y con logs sin buffer (se ven en `docker logs`)
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1

WORKDIR /app

# Dependencias primero: aprovecha la caché de capas
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip \\
 && pip install --no-cache-dir -r requirements.txt

# Artefactos del modelo y tabla de referencia
COPY modelo_binario_final.pkl  .
COPY features_binario.pkl      .
COPY encoder_binario.pkl       .
COPY umbral_optimo.pkl         .
COPY metadata.json             .
COPY municipios_features.csv   .

# API
COPY app.py .

# Usuario sin privilegios
RUN useradd --create-home --uid 1000 apiuser && chown -R apiuser:apiuser /app
USER apiuser

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --start-period=20s --retries=3 \\
  CMD python -c "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://localhost:8000/salud').status==200 else 1)"

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
with open(RUTA_DOCKER / "Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile)

dockerignore = """__pycache__/
*.pyc
*.ipynb
.ipynb_checkpoints/
.git/
.gitignore
README.md
test_api.py
"""
with open(RUTA_DOCKER / ".dockerignore", "w", encoding="utf-8") as f:
    f.write(dockerignore)

print(f"✓ Dockerfile (python:{PY_TAG}-slim) y .dockerignore  →  {RUTA_DOCKER}")




✓ Dockerfile (python:3.13-slim) y .dockerignore  →  /content/drive/MyDrive/Máster Data Science/TFM/docker


**README.md generado desde la metadata**

In [86]:
m  = metadata
mt = m["metricas_test"]

readme = f"""# API — Predicción de Riesgo de Desnutrición Infantil en Colombia

**TFM · Máster en Data Science · Jefferson Montoya Hoyos**

Servicio REST que clasifica municipios de Colombia como `prioritario` o
`no_prioritario` según el riesgo de desnutrición aguda en menores de 5 años.

---

## Requisitos

- Docker Desktop: https://www.docker.com/products/docker-desktop
- Nada más: la imagen instala todas las dependencias.

---

## Uso

### 1. Construir la imagen (solo la primera vez)

    docker build -t tfm-desnutricion .

### 2. Levantar la API

    docker run -d -p 8000:8000 --name api-desnutricion tfm-desnutricion

### 3. Documentación interactiva

    http://localhost:8000/docs

### 4. Comprobar que responde bien

    python test_api.py

### 5. Parar y limpiar

    docker stop api-desnutricion
    docker rm api-desnutricion

---

## Endpoints

| Método | Ruta               | Descripción                                    |
|--------|--------------------|------------------------------------------------|
| GET    | `/`                | Estado de la API                               |
| GET    | `/salud`           | Health check, métricas y versiones del modelo  |
| GET    | `/features`        | Lista y orden exacto de las features           |
| GET    | `/municipios`      | Municipios disponibles (filtrable por año)     |
| GET    | `/ranking?año=`    | Ranking nacional de riesgo para un año         |
| POST   | `/predecir`        | Predicción para un municipio (cod_mun + año)   |
| POST   | `/predecir/lote`   | Predicción para varios municipios (máx. 500)   |
| POST   | `/predecir/manual` | Simulación what-if con las features a mano     |

---

## Modelo

- **Algoritmo:** {m['modelo']} (`{m['clase']}`)
- **Hiperparámetros:** `n_estimators={m['hiperparametros'].get('n_estimators')}`,
  `max_depth={m['hiperparametros'].get('max_depth')}`,
  `min_samples_leaf={m['hiperparametros'].get('min_samples_leaf')}`,
  `class_weight={m['hiperparametros'].get('class_weight')}`
- **Umbral de decisión:** {m['umbral_decision']}
- **Features:** {mt['n_features']} variables — indicadores IPM del Censo 2018,
  topografía y clima (altitud, piso térmico, área), distancias a centros
  urbanos, rezagos temporales de la tasa y MDM del DNP
- **Entrenamiento:** SIVIGILA {mt['años_train'][0]}–{mt['años_train'][-1]} ({mt['n_train']:,} observaciones municipio-año)
- **Test:** SIVIGILA {mt['años_test'][0]}–{mt['años_test'][-1]} ({mt['n_test']:,} observaciones), split temporal estricto
- **Validación cruzada:** StratifiedGroupKFold (5 folds) agrupada por municipio

### Resultados en test

| Métrica                | Valor |
|------------------------|-------|
| Recall (prioritario)   | {mt['recall']} |
| Especificidad          | {mt['especificidad']} |
| F1 (prioritario)       | {mt['f1']} |
| F1-macro               | {mt['f1_macro']} |
| AUC-ROC                | {mt['auc_roc']} |

Matriz de confusión: {mt['matriz_confusion']['prioritarios_detectados']} prioritarios
detectados y {mt['matriz_confusion']['falsos_negativos']} omitidos;
{mt['matriz_confusion']['no_prioritario_correctos']} no prioritarios correctos y
{mt['matriz_confusion']['falsos_positivos']} falsos positivos.

---

## Cómo entrega las predicciones

El modelo usa {mt['n_features']} variables, muchas derivadas (rezagos,
distancias, altitud). Pedírselas todas al usuario en cada petición no es
realista, así que la imagen incluye `municipios_features.csv` con una fila por
municipio-año ya construida: **la API recibe `cod_mun` y `año` y busca
internamente el vector completo de features.**

Cobertura: {m['cobertura_tabla']['municipios']:,} municipios ·
años {m['cobertura_tabla']['años'][0]}–{m['cobertura_tabla']['años'][-1]} ·
{m['cobertura_tabla']['filas']:,} filas.

Para escenarios hipotéticos existe `POST /predecir/manual`, que acepta las
features directamente. `GET /features` devuelve la lista exacta.

---

## Estructura

    docker/
    ├── app.py                     # Servidor FastAPI
    ├── Dockerfile                 # Imagen
    ├── .dockerignore
    ├── requirements.txt           # Dependencias con versiones fijadas
    ├── modelo_binario_final.pkl   # Modelo entrenado
    ├── features_binario.pkl       # Lista ordenada de features
    ├── encoder_binario.pkl        # LabelEncoder (no_prioritario / prioritario)
    ├── umbral_optimo.pkl          # Umbral de decisión ({m['umbral_decision']})
    ├── metadata.json              # Versiones, hiperparámetros y métricas
    ├── municipios_features.csv    # Tabla de referencia municipio-año
    ├── test_api.py                # Pruebas de humo
    └── README.md

---

## Ejemplo (Python)

    import requests

    r = requests.post(
        "http://localhost:8000/predecir",
        json={{"cod_mun": "44847", "año": {mt['años_test'][-1]}}},
    )
    print(r.json())

    # Ranking nacional
    print(requests.get(
        "http://localhost:8000/ranking",
        params={{"año": {mt['años_test'][-1]}, "top": 10}},
    ).json())

---

## Reproducibilidad

Entrenado con Python {m['versiones']['python']},
scikit-learn {m['versiones']['scikit-learn']},
numpy {m['versiones']['numpy']},
pandas {m['versiones']['pandas']}.
`requirements.txt` fija exactamente esas versiones: si se cambian, la
deserialización del `.pkl` puede fallar o devolver resultados distintos.

Artefactos generados el {m['entrenado_en']}.
"""

with open(RUTA_DOCKER / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)
print(f"✓ README.md  ({len(readme):,} caracteres)  →  {RUTA_DOCKER}")


# =============================================================================
# CELDA 10.6 — test_api.py (pruebas de humo)
# =============================================================================
test_code = '''"""Pruebas de humo de la API. Uso: python test_api.py"""
import sys

import requests

BASE = "http://localhost:8000"
ok, fallos = 0, []


def check(nombre, cond, detalle=""):
    global ok
    if cond:
        ok += 1
        print(f"  [OK]    {nombre}")
    else:
        fallos.append(nombre)
        print(f"  [FALLA] {nombre}  {detalle}")


print("Probando API en", BASE)

try:
    r = requests.get(f"{BASE}/salud", timeout=10)
except Exception as e:
    print(f"\\nNo se pudo conectar: {e}")
    print("¿Está el contenedor corriendo?   docker ps")
    print("Si no aparece:                   docker logs api-desnutricion")
    sys.exit(1)

check("GET /salud responde 200", r.status_code == 200, r.text[:200])
salud = r.json()
check("modelo cargado", "modelo" in salud)
check("umbral cargado", isinstance(salud.get("umbral_decision"), (int, float)))
print(f"         modelo={salud.get('modelo')}  features={salud.get('n_features')}  "
      f"umbral={salud.get('umbral_decision')}")

r = requests.get(f"{BASE}/features", timeout=10)
check("GET /features responde 200", r.status_code == 200)

anios = salud.get("años_disponibles") or []
anio = anios[-1] if anios else 2024

r = requests.get(f"{BASE}/municipios", params={"año": anio, "limite": 3}, timeout=10)
check("GET /municipios responde 200", r.status_code == 200)
muns = r.json().get("municipios", [])
check("hay municipios en la tabla", len(muns) > 0)

if muns:
    cod = muns[0]["cod_mun"]
    r = requests.post(f"{BASE}/predecir", json={"cod_mun": cod, "año": anio}, timeout=10)
    check("POST /predecir responde 200", r.status_code == 200, r.text[:300])
    if r.status_code == 200:
        p = r.json()
        check("probabilidad en [0,1]", 0.0 <= p["probabilidad"] <= 1.0, str(p.get("probabilidad")))
        check("clasificación válida",
              p["clasificacion"] in ("prioritario", "no_prioritario"), p.get("clasificacion"))
        print(f"         {cod} ({anio}) -> {p['clasificacion']} ({p['probabilidad']:.1%})")

    lote = [{"cod_mun": m["cod_mun"], "año": anio} for m in muns]
    r = requests.post(f"{BASE}/predecir/lote", json=lote, timeout=20)
    check("POST /predecir/lote responde 200", r.status_code == 200, r.text[:300])
    if r.status_code == 200:
        check("el lote devuelve el mismo nº de filas", r.json()["total"] == len(lote))

r = requests.get(f"{BASE}/ranking", params={"año": anio, "top": 5}, timeout=60)
check("GET /ranking responde 200", r.status_code == 200, r.text[:300])
if r.status_code == 200:
    top = r.json()["top"]
    check("el ranking viene ordenado",
          all(top[i]["probabilidad"] >= top[i + 1]["probabilidad"] for i in range(len(top) - 1)))

r = requests.post(f"{BASE}/predecir", json={"cod_mun": "99999", "año": anio}, timeout=10)
check("municipio inexistente devuelve 404", r.status_code == 404, f"devolvió {r.status_code}")

r = requests.post(f"{BASE}/predecir/manual", json={"valores": {"analfabetismo": 10.0}}, timeout=10)
check("escenario incompleto devuelve 422", r.status_code == 422, f"devolvió {r.status_code}")

print(f"\\n{ok} pruebas superadas, {len(fallos)} fallidas")
if fallos:
    print("Fallaron:", ", ".join(fallos))
sys.exit(1 if fallos else 0)
'''

with open(RUTA_DOCKER / "test_api.py", "w", encoding="utf-8") as f:
    f.write(test_code)
print(f"✓ test_api.py  →  {RUTA_DOCKER}")




✓ README.md  (4,581 caracteres)  →  /content/drive/MyDrive/Máster Data Science/TFM/docker
✓ test_api.py  →  /content/drive/MyDrive/Máster Data Science/TFM/docker


**Verificación:**

In [87]:
ESPERADOS = [
    "app.py", "Dockerfile", ".dockerignore", "requirements.txt", "README.md",
    "modelo_binario_final.pkl", "features_binario.pkl", "encoder_binario.pkl",
    "umbral_optimo.pkl", "metadata.json", "municipios_features.csv", "test_api.py",
]

print("=" * 64)
print("  VERIFICACIÓN — CARPETA DOCKER")
print("=" * 64)
print(f"  Ruta: {RUTA_DOCKER}\n")

problemas = []
for nombre in ESPERADOS:
    ruta = RUTA_DOCKER / nombre
    if ruta.exists():
        print(f"  ✓  {nombre:<32} {ruta.stat().st_size/1024:>9.1f} KB")
    else:
        problemas.append(f"falta {nombre}")
        print(f"  ✗  {nombre:<32}   FALTA")

# ── Simulacro: exactamente lo que hace app.py al arrancar ──
print("\n  Simulacro de arranque de la API...")
try:
    _mod = joblib.load(RUTA_DOCKER / "modelo_binario_final.pkl")
    _fea = joblib.load(RUTA_DOCKER / "features_binario.pkl")
    _enc = joblib.load(RUTA_DOCKER / "encoder_binario.pkl")
    _umb = float(joblib.load(RUTA_DOCKER / "umbral_optimo.pkl"))
    _met = json.loads((RUTA_DOCKER / "metadata.json").read_text(encoding="utf-8"))
    _tab = pd.read_csv(RUTA_DOCKER / "municipios_features.csv", dtype={"cod_mun": str})

    _sin = [c for c in _fea if c not in _tab.columns]
    assert not _sin, f"features ausentes en la tabla: {_sin}"
    assert list(_enc.classes_) == ["no_prioritario", "prioritario"], \
        f"orden de clases inesperado: {list(_enc.classes_)}"

    _X = _tab.iloc[[0]][_fea].astype(float)
    _p = float(_mod.predict_proba(_X)[0][1])
    assert 0.0 <= _p <= 1.0

    print(f"    ✓ Artefactos cargados y coherentes entre sí")
    print(f"    ✓ Umbral leído: {_umb}")
    print(f"    ✓ Predicción de prueba: {_p:.4f} "
          f"({'prioritario' if _p >= _umb else 'no_prioritario'})")

    # El modelo del .pkl debe coincidir con el de la sesión
    _p_sesion = float(MODELO_FINAL.predict_proba(_X)[0][1])
    assert abs(_p - _p_sesion) < 1e-9, "el .pkl no coincide con MODELO_FINAL"
    print(f"    ✓ El .pkl reproduce exactamente MODELO_FINAL")

except Exception as e:
    problemas.append(f"simulacro: {e}")
    print(f"    ✗ El simulacro falló: {e}")

if problemas:
    print(f"\n  ⚠ Pendiente: {problemas}")
else:
    print("\n  ✅ Todo listo. En tu PC, dentro de la carpeta docker/:\n")
    print("     docker build -t tfm-desnutricion .")
    print("     docker run -d -p 8000:8000 --name api-desnutricion tfm-desnutricion")
    print("     python test_api.py")
    print("     http://localhost:8000/docs")

  VERIFICACIÓN — CARPETA DOCKER
  Ruta: /content/drive/MyDrive/Máster Data Science/TFM/docker

  ✓  app.py                                10.1 KB
  ✓  Dockerfile                             1.0 KB
  ✓  .dockerignore                          0.1 KB
  ✓  requirements.txt                       0.1 KB
  ✓  README.md                              4.6 KB
  ✓  modelo_binario_final.pkl            2016.5 KB
  ✓  features_binario.pkl                   0.7 KB
  ✓  encoder_binario.pkl                    0.4 KB
  ✓  umbral_optimo.pkl                      0.0 KB
  ✓  metadata.json                          2.3 KB
  ✓  municipios_features.csv             1064.2 KB
  ✓  test_api.py                            3.1 KB

  Simulacro de arranque de la API...
    ✓ Artefactos cargados y coherentes entre sí
    ✓ Umbral leído: 0.5
    ✓ Predicción de prueba: 0.2007 (no_prioritario)
    ✓ El .pkl reproduce exactamente MODELO_FINAL

  ✅ Todo listo. En tu PC, dentro de la carpeta docker/:

     docker build -t tfm